In [2]:
library(data.table)
library(stringdist)
library(ggplot2)
library(ggpubr)
# library(scales)
library(tidyverse)
library(seqinr)
# library(DNABarcodes)

## 1. split allele table into individual clones (in phy format)

In [6]:
mice=c("Q25","Q186","Q192","C4026","C4007","C4011","C4033")
samplesAmplicon = list(c("LLT-1","LLT-2","LvT-1-1","LvT-1-2","LvT-3","LvT-5","LvT-6","RLT","ThyT"),
    c("LLT","LvT-1","LvT-2"),
    c("LLT","LvT"),
    c("LLT","LvT-2","LvT-3","LvT-4","OvaT","RLT","ThyT"),
    c("LLT-1","LLT-2","LvT-1","LvT-2","OvaT","RLT","ThyT","IntT"),
    c("LLT-1","LLT-2","LOvaT","LvT-1","LvT-2","LvT-3","LvT-4","RLT","ROvaT","SpT","ThyT","IntT"),
    c("LLT-1","LLT-2","LVs","LvT","RLT","Spt","ThyT"))
PacBioDIR="/syn2/zhaolian/3.JiLab/results/3.PacBio/2.alleleTable/"
AmpliconDIR="/syn2/zhaolian/3.JiLab/results/1.BarcodeSeq/04.clone/"
for(my.mouse in 1:7){
    message("Calculating ",mice[my.mouse],"...")
    ########################################################################
    OUTDIR=paste0("/syn2/zhaolian/3.JiLab/results/3.PacBio/3.clones/",mice[my.mouse],"/")
    if (!dir.exists(OUTDIR)){
      dir.create(OUTDIR)
    }
    ########################################################################
    ## 1. Amplicon, reshape clone info
    clones=read.delim(paste0(AmpliconDIR,mice[my.mouse],"_assigned_clones_clean.txt"),header = T)
    setDT(clones)
    clones$sample=sapply(strsplit(clones$cellBC,"_"),'[',1)
    clones$cell=sapply(strsplit(clones$cellBC,"_"),'[',2)
    clones$lg=sapply(strsplit(clones$cellBC,"_"),'[',3)
    clones[,sample:=paste0(mice[my.mouse],"_",sample)]
    clones[,cellBC:=paste0(sample,"_",cell)]
    # clones[1:4,]
    setDF(clones)
    df_clones=data.table()
    intBCs=colnames(clones)[2:(ncol(clones)-3)]
    for(my_intBC in 1:length(intBCs)){
        temp <- clones[,c(1,my_intBC+1,ncol(clones)-2,ncol(clones))]
        colnames(temp)[2]="count"
        temp$intBC=intBCs[my_intBC]
        df_clones <- rbind(df_clones, temp[temp$count>0,])
    }
    df_clones[,count:=NULL]
    df_clones[,cellBC_intBC:= paste0(cellBC,"_",intBC)]
    numCellInClones=length(unique(df_clones$cellBC))
    df_clones <- df_clones[,c(5,3)]

    ########################################################################
    subs=samplesAmplicon[[my.mouse]]
    dt <- data.table()
    for(my.sample in subs){
        ########################################################################
        ## 2. PacBio
        allele_table=read.delim(paste0(PacBioDIR,mice[my.mouse],"_",my.sample,"_allele-table.txt"),header=F,
                   colClasses=c("character","character","character","numeric","numeric","character","character"),
                   col.names = c("cellBC","intBC","sample","count","readCount","nuc","bi"))
        setDT(allele_table)
        allele_table[,cellBC:=paste0(sample,"_",cellBC)]
        dt <- rbind(dt, allele_table)
    }
    dt[,cellBC_intBC:= paste0(cellBC,"_",intBC)]
    dt <- dt[cellBC_intBC %in% df_clones$cellBC_intBC]
    dt <- merge(df_clones, dt, by="cellBC_intBC",all=F)
    write.table(dt, file=paste0("/syn2/zhaolian/3.JiLab/results/3.PacBio/3.clones/",mice[my.mouse],"_pacBio_asnClones.txt"),sep ="\t", row.names=F, quote=F)

    message(length(unique(dt$cellBC))," out of ",numCellInClones," cells detected by PacBio")
    ########################################################################
    ########################################################################
    for(my.intBC in unique(dt$intBC)){
        temp <- dt[intBC == my.intBC]
        if(nrow(temp) != length(unique(temp$cellBC)) | length(unique(temp$lg))!=1){
            print(my.intBC)
        }
        ##
        s1 <- temp[,c(1,8)]
        colnames(s1) <- c("name","seq")
        s1$name <- gsub(paste0("_",my.intBC),"",s1$name)
        write.fasta(sequences=as.list(s1$seq), names=s1$name,file.out=paste0(OUTDIR,"Clone",temp$lg[1],"_",temp$intBC[1],".fasta"))
        ##
        s2 <- temp[,c(1,9)]
        colnames(s2) <- c("name","seq")
        s2$name <- gsub(paste0("_",my.intBC),"",s2$name)
        ref <- data.table(name="ref",seq=paste(rep("0",476),collapse = ""))
        s2 <- rbind(ref,s2)
        colnames(s2) <- c(as.character(nrow(s2)),"476")
        write.table(s2, file = paste0(OUTDIR,"Clone",temp$lg[1],"_",temp$intBC[1],".phy"),sep=" ",quote=F, row.names = F)
    }
}


Calculating Q25...

19719 out of 28918 cells detected by PacBio

Calculating Q186...

5544 out of 8927 cells detected by PacBio

Calculating Q192...

6255 out of 9424 cells detected by PacBio

Calculating C4026...

2877 out of 3817 cells detected by PacBio

Calculating C4007...

19120 out of 41773 cells detected by PacBio

Calculating C4011...

12203 out of 34212 cells detected by PacBio

Calculating C4033...

17014 out of 23621 cells detected by PacBio

